# LeetCode 3134 — Find the Median of the Uniqueness Array bằng Python

- Link: https://leetcode.com/problems/find-the-median-of-the-uniqueness-array/
- Độ khó: **Hard**
- Chủ đề: **Tìm nhị phân trên đáp án** + **cửa sổ trượt đếm số mảng con có ≤ k giá trị khác nhau**; trung vị = giá trị `k` nhỏ nhất mà số mảng con có `distinct ≤ k` đạt `⌈T/2⌉` (với `T = n(n+1)/2`)

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def medianOfUniquenessArray(self, nums: List[int]) -> int:
```

## 1. Mục tiêu học

1. "Mảng độc nhất" có `T = n(n+1)/2` phần tử — không thể liệt kê khi `n = 10^5` (`5·10⁹` phần tử).
2. Trung vị (lấy phần tử giữa **nhỏ hơn** khi chẵn) là phần tử thứ `⌈T/2⌉` (1-index) của dãy đã sắp ⇒ `k` nhỏ nhất với `count(≤ k) ≥ ⌈T/2⌉`.
3. `count(≤ k)` = số mảng con có nhiều nhất `k` giá trị khác nhau — cửa sổ trượt `O(n)` (LC 992 / 340).
4. `count(≤ k)` không giảm theo `k` ⇒ tìm nhị phân `k ∈ [1, n]` ⇒ `O(n log n)`.
5. Kiểm chứng bằng brute liệt kê và sắp xếp.

## 2. Đề bài & ví dụ

Mảng độc nhất của `nums` là mảng **đã sắp** chứa `distinct(nums[i..j])` cho mọi `0 ≤ i ≤ j < n`. Trả về **trung vị** của nó (nếu có hai phần tử giữa, lấy cái **nhỏ hơn**).

**Ví dụ 1:** `[1,2,3]` → `1` (mảng độc nhất `[1,1,1,2,2,3]`). **Ví dụ 2:** `[3,4,3,4,5]` → `2`. **Ví dụ 3:** `[4,3,5,4]` → `2`.

**Ràng buộc:** `1 ≤ n ≤ 10^5`; `1 ≤ nums[i] ≤ 10^5`.

In [1]:
from typing import List
from collections import defaultdict
import random, time

examples = [([1, 2, 3], 1), ([3, 4, 3, 4, 5], 2), ([4, 3, 5, 4], 2)]
for a, exp in examples:
    print(a, '-> kỳ vọng', exp)

[1, 2, 3] -> kỳ vọng 1
[3, 4, 3, 4, 5] -> kỳ vọng 2
[4, 3, 5, 4] -> kỳ vọng 2


## 3. Kiến thức nền

### 3.1. Vị trí trung vị

Dãy sắp `u_1 ≤ … ≤ u_T`. Nếu `T` lẻ, trung vị là `u_{(T+1)/2}`; nếu chẵn, đề lấy cái nhỏ hơn `u_{T/2}`. Cả hai đều là `u_{⌈T/2⌉}`… — cụ thể `⌈T/2⌉ = (T + 1) // 2`. Và `u_p ≤ k` ⇔ có ít nhất `p` phần tử `≤ k` ⇒ `u_p = min{ k : count(≤ k) ≥ p }`.

In [2]:
for T in range(1, 8):
    arr = list(range(1, T + 1))
    lower_median = arr[(T - 1) // 2]
    print(f'T = {T}: trung vị (nhỏ hơn khi chẵn) = phần tử thứ {lower_median} = (T + 1) // 2 = {(T + 1) // 2}')

T = 1: trung vị (nhỏ hơn khi chẵn) = phần tử thứ 1 = (T + 1) // 2 = 1
T = 2: trung vị (nhỏ hơn khi chẵn) = phần tử thứ 1 = (T + 1) // 2 = 1
T = 3: trung vị (nhỏ hơn khi chẵn) = phần tử thứ 2 = (T + 1) // 2 = 2
T = 4: trung vị (nhỏ hơn khi chẵn) = phần tử thứ 2 = (T + 1) // 2 = 2
T = 5: trung vị (nhỏ hơn khi chẵn) = phần tử thứ 3 = (T + 1) // 2 = 3
T = 6: trung vị (nhỏ hơn khi chẵn) = phần tử thứ 3 = (T + 1) // 2 = 3
T = 7: trung vị (nhỏ hơn khi chẵn) = phần tử thứ 4 = (T + 1) // 2 = 4


### 3.2. Đếm mảng con có `≤ k` giá trị khác nhau (cửa sổ trượt)

Với mỗi đầu phải `r`, kéo đầu trái `l` sang phải cho đến khi cửa sổ `[l, r]` có `≤ k` giá trị khác nhau; mọi mảng con `[l', r]` với `l' ≥ l` đều hợp lệ ⇒ cộng `r − l + 1`. Mỗi chỉ số vào/ra cửa sổ một lần ⇒ `O(n)`.

In [3]:
def count_at_most(nums, k):
    freq = defaultdict(int); distinct = 0; l = 0; total = 0
    for r, x in enumerate(nums):
        if freq[x] == 0:
            distinct += 1
        freq[x] += 1
        while distinct > k:
            y = nums[l]; freq[y] -= 1
            if freq[y] == 0:
                distinct -= 1
            l += 1
        total += r - l + 1
    return total

a = [3, 4, 3, 4, 5]
print('count(≤ k) cho k = 1..3:', [count_at_most(a, k) for k in range(1, 4)], '| T =', len(a) * (len(a) + 1) // 2)

count(≤ k) cho k = 1..3: [5, 12, 15] | T = 15


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Không cần dựng mảng độc nhất — chỉ cần trả lời câu hỏi "có bao nhiêu phần tử `≤ k`?" thật nhanh. Câu hỏi này đơn điệu theo `k` ⇒ tìm nhị phân `k`.

Ví dụ 2: `T = 15`, cần phần tử thứ `8`. `count(≤ 1) = 5 < 8`, `count(≤ 2) = 12 ≥ 8` ⇒ trung vị `2`.

## 5. Cách 1 — Brute Force: liệt kê, sắp xếp

In [4]:
def brute(nums):
    n = len(nums); vals = []
    for i in range(n):
        seen = set()
        for j in range(i, n):
            seen.add(nums[j]); vals.append(len(seen))
    vals.sort()
    return vals[(len(vals) - 1) // 2]

for a, exp in examples:
    print('brute:', brute(a), '| kỳ vọng', exp)

brute: 1 | kỳ vọng 1
brute: 2 | kỳ vọng 2
brute: 2 | kỳ vọng 2


**Độ phức tạp:** `O(n² log n)` thời gian, `O(n²)` bộ nhớ.

## 6. Ý tưởng thuật toán / bước đột phá

```
T = n(n+1)/2; need = (T + 1) // 2
lo, hi = 1, số giá trị khác nhau của nums
while lo < hi:
    mid = (lo + hi) // 2
    if count_at_most(nums, mid) >= need: hi = mid
    else: lo = mid + 1
return lo
```

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Đếm theo giá trị `distinct` bằng `O(n²)` (không sắp xếp)

Với mỗi `i`, duyệt `j` tăng dần, tăng `cnt[distinct]`; cuối cùng cộng dồn `cnt` để tìm phần tử thứ `need`. Bộ nhớ `O(n)`.

In [5]:
def counting_n2(nums):
    n = len(nums); cnt = [0] * (n + 2)
    for i in range(n):
        seen = set()
        for j in range(i, n):
            seen.add(nums[j]); cnt[len(seen)] += 1
    T = n * (n + 1) // 2; need = (T + 1) // 2; acc = 0
    for k in range(1, n + 1):
        acc += cnt[k]
        if acc >= need:
            return k

for a, exp in examples:
    print('đếm O(n²):', counting_n2(a), '| kỳ vọng', exp)

đếm O(n²): 1 | kỳ vọng 1
đếm O(n²): 2 | kỳ vọng 2
đếm O(n²): 2 | kỳ vọng 2


### 7.2. Cách 3 — Tìm nhị phân + cửa sổ trượt (lời giải chính)

In [6]:
def binary_search(nums):
    n = len(nums); T = n * (n + 1) // 2; need = (T + 1) // 2
    lo, hi = 1, len(set(nums))
    while lo < hi:
        mid = (lo + hi) // 2
        if count_at_most(nums, mid) >= need:
            hi = mid
        else:
            lo = mid + 1
    return lo

for a, exp in examples:
    print('tìm nhị phân:', binary_search(a), '| kỳ vọng', exp)

tìm nhị phân: 1 | kỳ vọng 1
tìm nhị phân: 2 | kỳ vọng 2
tìm nhị phân: 2 | kỳ vọng 2


### 7.3. Cách 4 — Cửa sổ trượt dừng sớm + mảng tần suất (nhanh hơn trong Python)

Hai tối ưu: (1) dùng **mảng** tần suất (giá trị `≤ 10^5`) thay cho `dict`; (2) dừng đếm ngay khi tổng đã `≥ need` (không cần biết tổng chính xác).

In [7]:
def binary_search_fast(nums):
    n = len(nums); T = n * (n + 1) // 2; need = (T + 1) // 2
    V = max(nums) + 1
    def enough(k):
        freq = [0] * V; distinct = 0; l = 0; total = 0
        for r in range(n):
            x = nums[r]
            if freq[x] == 0:
                distinct += 1
            freq[x] += 1
            while distinct > k:
                y = nums[l]; freq[y] -= 1
                if freq[y] == 0:
                    distinct -= 1
                l += 1
            total += r - l + 1
            if total >= need:
                return True
        return False
    lo, hi = 1, len(set(nums))
    while lo < hi:
        mid = (lo + hi) // 2
        if enough(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo

for a, exp in examples:
    print('tìm nhị phân nhanh:', binary_search_fast(a), '| kỳ vọng', exp)

tìm nhị phân nhanh: 1 | kỳ vọng 1
tìm nhị phân nhanh: 2 | kỳ vọng 2
tìm nhị phân nhanh: 2 | kỳ vọng 2


## 8. TRACE từng bước

Trace tìm nhị phân trên `nums = [1, 2, 1, 3, 2, 4, 1, 5]`.

In [8]:
def trace(nums):
    n = len(nums); T = n * (n + 1) // 2; need = (T + 1) // 2
    lo, hi = 1, len(set(nums))
    print(f'T = {T}, cần phần tử thứ {need}, khoảng tìm [{lo}, {hi}]')
    while lo < hi:
        mid = (lo + hi) // 2; c = count_at_most(nums, mid)
        print(f'  mid = {mid}: count(≤ {mid}) = {c} -> {"đủ, hi = mid" if c >= need else "thiếu, lo = mid + 1"}')
        if c >= need:
            hi = mid
        else:
            lo = mid + 1
    print('trung vị =', lo, '| brute:', brute(nums))

trace([1, 2, 1, 3, 2, 4, 1, 5])

T = 36, cần phần tử thứ 18, khoảng tìm [1, 5]
  mid = 3: count(≤ 3) = 24 -> đủ, hi = mid
  mid = 2: count(≤ 2) = 16 -> thiếu, lo = mid + 1
trung vị = 3 | brute: 3


## 9. Code nộp LeetCode

In [9]:
class Solution:
    def medianOfUniquenessArray(self, nums: List[int]) -> int:
        n = len(nums)
        need = (n * (n + 1) // 2 + 1) // 2
        V = max(nums) + 1

        def enough(k):
            freq = [0] * V
            distinct = 0
            l = 0
            total = 0
            for r in range(n):
                x = nums[r]
                if freq[x] == 0:
                    distinct += 1
                freq[x] += 1
                while distinct > k:
                    y = nums[l]
                    freq[y] -= 1
                    if freq[y] == 0:
                        distinct -= 1
                    l += 1
                total += r - l + 1
                if total >= need:
                    return True
            return False

        lo, hi = 1, len(set(nums))
        while lo < hi:
            mid = (lo + hi) // 2
            if enough(mid):
                hi = mid
            else:
                lo = mid + 1
        return lo

for a, exp in examples:
    got = Solution().medianOfUniquenessArray(a)
    print(got, 'OK' if got == exp else 'SAI')

1 OK
2 OK
2 OK


## 10. Vì sao thuật toán đúng

1. **Trung vị = phần tử thứ `need = ⌈T/2⌉`** của dãy đã sắp (mục 3.1).
2. **Đơn điệu:** `count(≤ k)` không giảm theo `k`; `count(≤ D) = T` với `D` = số giá trị khác nhau của mảng ⇒ đáp án thuộc `[1, D]` và là `k` nhỏ nhất với `count(≤ k) ≥ need`.
3. **Cửa sổ trượt:** "có `≤ k` giá trị khác nhau" đóng với việc thu hẹp; với mỗi `r`, `l` nhỏ nhất hợp lệ không giảm theo `r` ⇒ số mảng con hợp lệ kết thúc tại `r` là `r − l + 1`.
4. **Dừng sớm** chỉ trả lời câu hỏi "đủ chưa" — đúng thứ tìm nhị phân cần.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Liệt kê + sắp | `O(n² log n)` | `O(n²)` |
| 2. Đếm `O(n²)` | `O(n²)` | `O(n)` |
| 3. Tìm nhị phân + cửa sổ | `O(n log D)` | `O(n)` |
| 4. Như 3 + dừng sớm + mảng | `O(n log D)` (hằng số nhỏ) | `O(V)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [10]:
impls = {'counting': counting_n2, 'binary': binary_search, 'fast': binary_search_fast,
         'submit': lambda a: Solution().medianOfUniquenessArray(a)}
edge = [([7], 1), ([5, 5], 1), ([1, 2], 1), ([1, 2, 3, 4], 2), ([1, 1, 1, 2, 2, 2], 1), ([1, 2, 3, 4, 5, 6, 7], 3)]
for a, exp in examples + edge:
    assert brute(a) == exp, (a, brute(a))
    for name, f in impls.items():
        assert f(a) == exp, (name, a)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


Giải thích: `[1, 2]` — mảng độc nhất `[1, 1, 2]`, trung vị `1`. `[1, 2, 3, 4]`: `T = 10`, mười giá trị `1,1,1,1,2,2,2,3,3,4` ⇒ phần tử thứ 5 là `2`.

### 12.2. Đối chiếu ngẫu nhiên với brute

In [11]:
rng = random.Random(3134)
cnt = 0
for _ in range(400):
    n = rng.randint(1, 30); a = [rng.randint(1, rng.choice([2, 4, 10, 30])) for _ in range(n)]
    exp = brute(a)
    for name, f in impls.items():
        assert f(a) == exp, (name, a)
    cnt += 1
print('Khớp brute trên', cnt, 'mảng ngẫu nhiên.')

Khớp brute trên 400 mảng ngẫu nhiên.


### 12.3. Ca lớn `n = 10^5`

In [12]:
N = 10 ** 5
big = {
    'ngẫu nhiên 1..10^5': [rng.randint(1, 10 ** 5) for _ in range(N)],
    'toàn khác nhau': list(range(1, N + 1)),
    'ít giá trị (1..50)': [rng.randint(1, 50) for _ in range(N)],
}
for name, a in big.items():
    t = time.time(); x = Solution().medianOfUniquenessArray(a); t1 = time.time() - t
    print(f'{name:22s}: trung vị = {x:6d} | nộp {t1:.2f}s')

ngẫu nhiên 1..10^5    : trung vị =  25348 | nộp 0.28s


toàn khác nhau        : trung vị =  29290 | nộp 0.25s
ít giá trị (1..50)    : trung vị =     50 | nộp 0.07s


Với mảng toàn khác nhau, `distinct(i..j) = j − i + 1` ⇒ số mảng con độ dài `≤ k` là `Σ_{L ≤ k} (n − L + 1)`; trung vị khoảng `n(1 − 1/√2) ≈ 0.293 n`. Ô dưới kiểm công thức này.

In [13]:
n = N; need = (n * (n + 1) // 2 + 1) // 2
acc = 0
for L in range(1, n + 1):
    acc += n - L + 1
    if acc >= need:
        break
assert Solution().medianOfUniquenessArray(list(range(1, n + 1))) == L
print('mảng toàn khác nhau: trung vị =', L, f'≈ {L / n:.3f} n')

mảng toàn khác nhau: trung vị = 29290 ≈ 0.293 n


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Lấy phần tử giữa **lớn hơn** khi `T` chẵn (`need = T // 2 + 1`)

In [14]:
def wrong_upper_median(nums):
    n = len(nums); T = n * (n + 1) // 2; need = T // 2 + 1                 # SAI khi T chẵn
    lo, hi = 1, len(set(nums))
    while lo < hi:
        mid = (lo + hi) // 2
        if count_at_most(nums, mid) >= need: hi = mid
        else: lo = mid + 1
    return lo

diff = next(a for a in ([rng.randint(1, 3) for _ in range(rng.randint(2, 8))] for _ in range(3000)) if wrong_upper_median(a) != brute(a))
print('nums =', diff, '| T =', len(diff) * (len(diff) + 1) // 2, '| sai :', wrong_upper_median(diff), '| đúng:', Solution().medianOfUniquenessArray(diff))

nums = [3, 1, 2] | T = 6 | sai : 2 | đúng: 1


**Vì sao sai:** khi `T` chẵn và hai phần tử giữa khác nhau, đề lấy cái **nhỏ hơn** (phần tử thứ `T/2`), còn `T // 2 + 1` lấy cái lớn hơn. Nhiều mảng (như ví dụ 1, hay `[1, 2, 3, 4]`) có hai phần tử giữa bằng nhau nên không lộ lỗi — phản ví dụ trên được tìm tự động.

### Lỗi 2 — Đếm `count(≤ k)` bằng cửa sổ nhưng cộng `1` thay vì `r − l + 1`

In [15]:
def wrong_window_count(nums, k):
    freq = defaultdict(int); distinct = 0; l = 0; total = 0
    for r, x in enumerate(nums):
        if freq[x] == 0: distinct += 1
        freq[x] += 1
        while distinct > k:
            y = nums[l]; freq[y] -= 1
            if freq[y] == 0: distinct -= 1
            l += 1
        total += 1                                                            # SAI: chỉ đếm cửa sổ dài nhất
    return total

def wrong_median_2(nums):
    n = len(nums); need = (n * (n + 1) // 2 + 1) // 2; lo, hi = 1, len(set(nums))
    while lo < hi:
        mid = (lo + hi) // 2
        if wrong_window_count(nums, mid) >= need: hi = mid
        else: lo = mid + 1
    return lo

print('sai :', wrong_median_2([1, 2, 3]), '| đúng:', Solution().medianOfUniquenessArray([1, 2, 3]))
print('sai :', wrong_median_2([1, 2, 3, 4, 5, 6, 7]), '| đúng:', Solution().medianOfUniquenessArray([1, 2, 3, 4, 5, 6, 7]))

sai : 1 | đúng: 1
sai : 7 | đúng: 3


**Vì sao sai:** với mỗi `r`, **mọi** đầu trái `l' ∈ [l, r]` đều hợp lệ — không chỉ cửa sổ dài nhất.

### Lỗi 3 — Tìm nhị phân trả về `lo` khi điều kiện là `count(≤ mid) > need` (ngặt)

In [16]:
def wrong_strict(nums):
    n = len(nums); need = (n * (n + 1) // 2 + 1) // 2; lo, hi = 1, len(set(nums))
    while lo < hi:
        mid = (lo + hi) // 2
        if count_at_most(nums, mid) > need: hi = mid                          # SAI: phải >=
        else: lo = mid + 1
    return lo

diff = next(a for a in ([rng.randint(1, 4) for _ in range(rng.randint(2, 8))] for _ in range(3000)) if wrong_strict(a) != brute(a))
print('nums =', diff, '| sai :', wrong_strict(diff), '| đúng:', Solution().medianOfUniquenessArray(diff))

nums = [3, 3, 3, 2, 1] | sai : 2 | đúng: 1


**Vì sao sai:** khi `count(≤ k)` **đúng bằng** `need`, phần tử thứ `need` chính là `k` — điều kiện ngặt bỏ lỡ và trả `k + 1`.

### Lỗi 4 — Trung vị của các **giá trị khác nhau** của `distinct` (bỏ qua bội số)

In [17]:
def wrong_unique_values(nums):
    n = len(nums); vals = set()
    for i in range(n):
        seen = set()
        for j in range(i, n):
            seen.add(nums[j]); vals.add(len(seen))
    v = sorted(vals)
    return v[(len(v) - 1) // 2]                                               # SAI

print('sai :', wrong_unique_values([1, 2, 3, 4, 5]), '| đúng:', Solution().medianOfUniquenessArray([1, 2, 3, 4, 5]))

sai : 3 | đúng: 2


**Vì sao sai:** mảng độc nhất có **lặp** — giá trị nhỏ xuất hiện nhiều lần hơn (nhiều mảng con ngắn), kéo trung vị xuống.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Phần tử thứ `p` / trung vị của một tập khổng lồ được định nghĩa ngầm" ⇒ tìm nhị phân giá trị + đếm `≤ k`.
- "Số mảng con có `≤ k` giá trị khác nhau" ⇒ cửa sổ trượt cộng `r − l + 1` (LC 992: `= k` là hiệu hai lần `≤`).
- Dừng sớm khi chỉ cần so với ngưỡng.

## 15. Biến thể của bài

1. **LC 992:** đếm mảng con có **đúng** `k` giá trị khác nhau ⇒ `count(≤ k) − count(≤ k−1)`.
2. Phần tử thứ `p` bất kỳ (không chỉ trung vị) ⇒ đổi `need`.
3. Trung vị của **tổng** mảng con (LC 1508-kiểu) ⇒ tìm nhị phân + hai con trỏ trên tổng tiền tố (với số không âm).
4. Trung vị của `max − min` các mảng con ⇒ tìm nhị phân + deque.

In [18]:
# Biến thể 1 (LC 992) và 2 (phần tử thứ p)
def exactly_k(nums, k):
    return count_at_most(nums, k) - (count_at_most(nums, k - 1) if k > 1 else 0)

def kth_of_uniqueness(nums, p):
    lo, hi = 1, len(set(nums))
    while lo < hi:
        mid = (lo + hi) // 2
        if count_at_most(nums, mid) >= p: hi = mid
        else: lo = mid + 1
    return lo

for _ in range(200):
    a = [rng.randint(1, 4) for _ in range(rng.randint(1, 12))]; n = len(a)
    vals = sorted(len(set(a[i:j + 1])) for i in range(n) for j in range(i, n))
    for k in range(1, 5):
        assert exactly_k(a, k) == vals.count(k)
    p = rng.randint(1, len(vals))
    assert kth_of_uniqueness(a, p) == vals[p - 1]
assert exactly_k([1, 2, 1, 2, 3], 2) == 7                     # ví dụ LC 992
print('Biến thể LC 992 và "phần tử thứ p" khớp brute.')

Biến thể LC 992 và "phần tử thứ p" khớp brute.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Liệt kê + sắp | `O(n² log n)` | Oracle |
| 2 | Đếm theo giá trị | `O(n²)` | Bộ nhớ `O(n)` |
| 3 | Tìm nhị phân + cửa sổ | `O(n log D)` | Lõi lời giải |
| 4 | Như 3 + tối ưu | `O(n log D)` | **Nộp bài** |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `[2, 2, 2]`.

**Bài 2 (tính tay):** `[1, 2, 1]` — viết mảng độc nhất.

**Bài 3 (điền TODO):** trả về **phần tử lớn hơn** trong hai phần tử giữa (trung vị "trên").

**Bài 4 (tự viết lại):** trung bình cộng của mảng độc nhất dưới dạng phân số `(tử, mẫu)`, `O(n)`.

**Bài 5 (mini challenge):** đếm tổng `Σ distinct(nums[i..j])` trên mọi mảng con trong `O(n)` (đóng góp của mỗi vị trí xuất hiện đầu tiên).

In [19]:
def upper_median_todo(nums):
    # TODO
    raise NotImplementedError

def mean_fraction_todo(nums):
    # TODO
    raise NotImplementedError

def total_distinct_todo(nums):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Mọi mảng con chỉ có một giá trị. (2) Mảng độc nhất toàn `1`. (3) `1`.

**Bài 2.** (1) Mảng con: `[1], [2], [1], [1,2], [2,1], [1,2,1]`. (2) `distinct`: `1, 1, 1, 2, 2, 2`. (3) Sắp: `[1, 1, 1, 2, 2, 2]`, trung vị `1`.

**Bài 3.** (1) Phần tử thứ `T // 2 + 1`. (2) Đổi `need`. (3) Kiểm brute.

**Bài 4.** (1) Tử = `Σ distinct` (bài 5). (2) Mẫu `T`. (3) Rút gọn bằng `gcd`.

**Bài 5.** (1) Mảng con `[i, j]` có `distinct` = số vị trí `p ∈ [i, j]` mà `nums[p]` không xuất hiện trong `[i, p)`. (2) Vị trí `p` với lần xuất hiện trước `prev[p]` đóng góp vào các `i ∈ (prev[p], p]`, `j ∈ [p, n)` ⇒ `(p − prev[p]) · (n − p)`. (3) Cộng lại.

## 19. Đáp án tham khảo cho bài tập

In [20]:
# Bài 1, 2, 3
assert Solution().medianOfUniquenessArray([2, 2, 2]) == 1
assert Solution().medianOfUniquenessArray([1, 2, 1]) == 1

def upper_median(nums):
    n = len(nums); T = n * (n + 1) // 2
    return kth_of_uniqueness(nums, T // 2 + 1)

for _ in range(200):
    a = [rng.randint(1, 5) for _ in range(rng.randint(1, 12))]; n = len(a)
    vals = sorted(len(set(a[i:j + 1])) for i in range(n) for j in range(i, n))
    assert upper_median(a) == vals[len(vals) // 2]
print('Bài 1: 1 | Bài 2: 1 | Bài 3: trung vị trên khớp brute.')

Bài 1: 1 | Bài 2: 1 | Bài 3: trung vị trên khớp brute.


In [21]:
# Bài 4, 5
from math import gcd
def total_distinct(nums):
    n = len(nums); last = {}; tot = 0
    for p, x in enumerate(nums):
        prev = last.get(x, -1)
        tot += (p - prev) * (n - p)
        last[x] = p
    return tot

def mean_fraction(nums):
    n = len(nums); T = n * (n + 1) // 2; s = total_distinct(nums); g = gcd(s, T)
    return s // g, T // g

for _ in range(200):
    a = [rng.randint(1, 5) for _ in range(rng.randint(1, 15))]; n = len(a)
    vals = [len(set(a[i:j + 1])) for i in range(n) for j in range(i, n)]
    assert total_distinct(a) == sum(vals)
    num, den = mean_fraction(a)
    assert num * len(vals) == sum(vals) * den and gcd(num, den) == 1
print('Bài 4 (trung bình dạng phân số) và Bài 5 (tổng distinct O(n)) khớp brute; ví dụ 2 ->', mean_fraction(examples[1][0]))

Bài 4 (trung bình dạng phân số) và Bài 5 (tổng distinct O(n)) khớp brute; ví dụ 2 -> (28, 15)


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
need = (n*(n+1)//2 + 1) // 2
lo, hi = 1, len(set(nums))
while lo < hi:
    mid = (lo + hi) // 2
    if at_most(mid) >= need: hi = mid        # cửa sổ trượt, cộng r - l + 1
    else: lo = mid + 1
return lo
```

**Checklist:**

- [ ] `need = ⌈T/2⌉` (trung vị **dưới** khi chẵn)?
- [ ] Cửa sổ cộng `r − l + 1` mỗi bước?
- [ ] Điều kiện tìm nhị phân `≥ need` (không ngặt)?
- [ ] `T` tới `5·10⁹` (Python không tràn; ngôn ngữ khác dùng 64-bit)?